# Reprodução didática: rede neural siamesa e inferência one-shot

Este notebook acompanha o seminário sobre **Siamese Neural Networks for One-shot Image Recognition** (Koch, Zemel e Salakhutdinov, 2015).

**Objetivo:** reproduzir os componentes centrais do método em um experimento pequeno e rápido:

- duas redes gêmeas com **pesos compartilhados**;
- treinamento por **verificação de pares** (mesma classe / classe diferente);
- comparação por **distância L1** entre embeddings;
- decisão por uma saída sigmoidal;
- uso da similaridade aprendida em uma tarefa **10-way one-shot**.

> **Importante:** este experimento usa o dataset `sklearn.datasets.load_digits` e classes já presentes no treinamento. Portanto, é uma **reprodução didática do mecanismo de inferência**, e não uma reprodução numérica do protocolo de classes inéditas do Omniglot usado no artigo.

In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('PyTorch:', torch.__version__)

PyTorch: 2.10.0+cpu


## 1. Dados

O conjunto `digits` contém imagens 8×8 de dígitos manuscritos. Separaremos amostras para treinamento, validação de verificação e avaliação one-shot.

In [2]:
X, y = load_digits(return_X_y=True)
X = (X / 16.0).astype('float32').reshape(-1, 1, 8, 8)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, stratify=y_train, random_state=SEED
)

print('Treino:', X_train.shape, 'Validação:', X_val.shape, 'Teste:', X_test.shape)

Treino: (1068, 1, 8, 8) Validação: (189, 1, 8, 8) Teste: (540, 1, 8, 8)


## 2. Geração de pares

A rede não é treinada diretamente para classificar 0–9. Ela recebe **dois exemplos** e aprende a responder se pertencem à mesma classe.

In [3]:
rng = np.random.default_rng(SEED)

def make_pairs(X, y, n_pairs, seed=SEED):
    local_rng = np.random.default_rng(seed)
    by_class = {c: np.where(y == c)[0] for c in np.unique(y)}
    classes = list(by_class)
    A, B, labels = [], [], []

    for i in range(n_pairs):
        if i % 2 == 0:  # par positivo
            c = int(local_rng.choice(classes))
            a, b = local_rng.choice(by_class[c], 2, replace=False)
            label = 1.0
        else:           # par negativo
            c1, c2 = local_rng.choice(classes, 2, replace=False)
            a = local_rng.choice(by_class[int(c1)])
            b = local_rng.choice(by_class[int(c2)])
            label = 0.0
        A.append(X[a]); B.append(X[b]); labels.append(label)

    return (
        torch.tensor(np.array(A)),
        torch.tensor(np.array(B)),
        torch.tensor(labels, dtype=torch.float32),
    )

A_tr, B_tr, Y_tr = make_pairs(X_train, y_train, 13_000, seed=42)
A_va, B_va, Y_va = make_pairs(X_val, y_val, 3_000, seed=43)

train_loader = DataLoader(TensorDataset(A_tr, B_tr, Y_tr), batch_size=256, shuffle=True)
val_loader = DataLoader(TensorDataset(A_va, B_va, Y_va), batch_size=512)
print('Pares de treino:', len(Y_tr), '| Pares de validação:', len(Y_va))

Pares de treino: 13000 | Pares de validação: 3000


## 3. Rede siamesa

Cada imagem passa pelo **mesmo extrator de características**. Depois calculamos `|h1 - h2|` e aprendemos uma combinação que produz a probabilidade de o par ser da mesma classe.

In [4]:
class SiameseNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 2 * 2, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
        )
        self.similarity = nn.Linear(64, 1)

    def forward(self, x1, x2):
        h1 = self.embedding(x1)
        h2 = self.embedding(x2)
        l1 = torch.abs(h1 - h2)
        return self.similarity(l1).squeeze(1)

model = SiameseNet()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss()
model

SiameseNet(
  (embedding): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=256, out_features=128, bias=True)
    (8): ReLU()
    (9): Linear(in_features=128, out_features=64, bias=True)
  )
  (similarity): Linear(in_features=64, out_features=1, bias=True)
)

## 4. Treinamento de verificação

In [5]:
history = []
for epoch in range(8):
    model.train()
    total_loss = 0.0
    n = 0
    for a, b, target in train_loader:
        optimizer.zero_grad()
        logits = model(a, b)
        loss = criterion(logits, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(target)
        n += len(target)

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for a, b, target in val_loader:
            pred = (torch.sigmoid(model(a, b)) >= 0.5).float()
            correct += (pred == target).sum().item()
            total += len(target)

    val_acc = correct / total
    history.append((epoch + 1, total_loss / n, val_acc))
    print(f'Época {epoch+1:02d} | perda={total_loss/n:.4f} | acurácia de verificação={val_acc:.4f}')

Época 01 | perda=0.6181 | acurácia de verificação=0.6947


Época 02 | perda=0.4190 | acurácia de verificação=0.8230


Época 03 | perda=0.2731 | acurácia de verificação=0.8690


Época 04 | perda=0.1899 | acurácia de verificação=0.8887


Época 05 | perda=0.1311 | acurácia de verificação=0.8917


Época 06 | perda=0.0889 | acurácia de verificação=0.8777


Época 07 | perda=0.0664 | acurácia de verificação=0.8907


Época 08 | perda=0.0488 | acurácia de verificação=0.9007


## 5. Inferência 10-way one-shot

Em cada episódio:

1. escolhemos **um único exemplo de suporte por classe**;
2. escolhemos uma imagem de consulta;
3. comparamos a consulta com os 10 exemplos;
4. escolhemos a classe com maior similaridade.

Não há retreinamento durante os episódios.

In [6]:
classes = sorted(np.unique(y_test))
by_class = {c: np.where(y_test == c)[0] for c in classes}
rng = np.random.default_rng(123)

N_TASKS = 1000
correct = 0
model.eval()

with torch.no_grad():
    for _ in range(N_TASKS):
        support_idx = {c: int(rng.choice(by_class[c])) for c in classes}
        query_class = int(rng.choice(classes))
        candidates = by_class[query_class][by_class[query_class] != support_idx[query_class]]
        q_idx = int(rng.choice(candidates))
        query = torch.tensor(X_test[q_idx:q_idx+1])

        scores = []
        for c in classes:
            support = torch.tensor(X_test[support_idx[c]:support_idx[c]+1])
            score = torch.sigmoid(model(query, support)).item()
            scores.append(score)

        pred_class = classes[int(np.argmax(scores))]
        correct += int(pred_class == query_class)

one_shot_acc = correct / N_TASKS
verification_acc = history[-1][2]
print(f'Acurácia final de verificação: {verification_acc:.3%}')
print(f'Acurácia 10-way one-shot:      {one_shot_acc:.3%}')
print('Baseline aleatória 10-way:     10.000%')

Acurácia final de verificação: 90.067%
Acurácia 10-way one-shot:      81.100%
Baseline aleatória 10-way:     10.000%


### Interpretação

O experimento mostra que a rede pode transformar uma tarefa de classificação em **comparações de similaridade**. Isso é a ideia central usada por Koch et al. (2015).

A comparação com os números do artigo deve ser feita com cuidado: o artigo usa Omniglot, classes inéditas e uma arquitetura/treinamento muito maiores. Neste notebook, o objetivo é apenas demonstrar o mecanismo de forma rápida e reproduzível para o seminário.

## 6. Como aproximar ainda mais o artigo

Para uma extensão no Colab:

- substituir `load_digits` por `torchvision.datasets.Omniglot`;
- separar alfabetos de background e evaluation;
- treinar em pares de caracteres do background;
- executar episódios 20-way no conjunto de evaluation;
- comparar com os **92,0%** reportados pela rede siamesa convolucional no artigo.

Isso exige mais tempo de treinamento e é melhor executado antes da apresentação.